# 📓 Course Module 3: Latent Predictive Learning (I-JEPA) & SSL Benchmark
Welcome to Module 3! In this final notebook, we explore two core topics:
1. **I-JEPA (Image Joint-Embedding Predictive Architecture):** Yann LeCun's vision for SSL that predicts in **representation space** rather than pixel space.
2. **Real Dataset Benchmark (CIFAR-10):** A complete, runnable experiment proving **how SSL outperforms supervised learning when labeled data is scarce** (e.g., using only 10% of labeled data).

---

## 💡 Part 1: What is I-JEPA?
Unlike Masked Autoencoders (MAE) which reconstruct pixel colors, **I-JEPA predicts the embeddings of target image blocks using context blocks**.

* **No pixel reconstruction:** Avoids wasting model capacity on fine details like background noise or individual pixels.
* **Predictor network:** Predicts target representations $s_y$ from context representation $s_x$.
* **Target Encoder (EMA):** Updated via Exponential Moving Average to provide stable target representations without collapsing.

![I-JEPA Architecture](https://raw.githubusercontent.com/facebookresearch/ijepa/main/.github/ijepa_architecture.png)
*(Image: Context patch is encoded into $s_x$. Target patches are encoded by a target encoder into $s_y$. A predictor predicts $s_y$ from $s_x$.)*

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### 1. SimpleJEPA Architecture Blueprint
Here is our complete PyTorch implementation of the Joint-Embedding Predictive Architecture.

In [ ]:
class SimpleJEPA(nn.Module):
    """
    Joint-Embedding Predictive Architecture (JEPA).
    - Context Encoder: Extracts representation s_x from context view.
    - Target Encoder: Extracts representation s_y from target view (EMA updated).
    - Predictor: Predicts s_y from s_x in representation space.
    """
    def __init__(self, embed_dim=512):
        super(SimpleJEPA, self).__init__()
        
        # 1. Context Encoder (Trainable Backbone)
        resnet_ctx = models.resnet18(weights=None)
        resnet_ctx.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False) # Optimized for CIFAR-10
        resnet_ctx.maxpool = nn.Identity()
        resnet_ctx.fc = nn.Identity()
        self.context_encoder = resnet_ctx
        
        # 2. Target Encoder (Updated via EMA)
        resnet_tgt = models.resnet18(weights=None)
        resnet_tgt.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        resnet_tgt.maxpool = nn.Identity()
        resnet_tgt.fc = nn.Identity()
        self.target_encoder = resnet_tgt
        
        # Freeze Target Encoder gradients
        for p in self.target_encoder.parameters():
            p.requires_grad = False
            
        # 3. Predictor Head (Predicts in representation space)
        self.predictor = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Linear(256, embed_dim)
        )
        
    def forward_context(self, x_ctx):
        s_x = self.context_encoder(x_ctx)
        pred_s_y = self.predictor(s_x)
        return pred_s_y
        
    @torch.no_grad()
    def forward_target(self, x_tgt):
        s_y = self.target_encoder(x_tgt)
        return s_y

    @torch.no_grad()
    def update_target_encoder(self, momentum=0.99):
        """ Exponential Moving Average (EMA) update for target encoder """
        for p_ctx, p_tgt in zip(self.context_encoder.parameters(), self.target_encoder.parameters()):
            p_tgt.data = momentum * p_tgt.data + (1 - momentum) * p_ctx.data

def jepa_loss(pred_s_y, s_y):
    """ Loss in latent representation space (Normalized Smooth L1 / Cosine Loss) """
    pred_s_y_norm = F.normalize(pred_s_y, dim=-1)
    s_y_norm = F.normalize(s_y, dim=-1)
    return F.smooth_l1_loss(pred_s_y_norm, s_y_norm)

---
## 🧪 Part 2: Real Dataset Comparison — SimpleJEPA SSL vs Supervised Baseline

Now we test **SimpleJEPA** directly on **CIFAR-10** to prove how SSL pre-training outperforms supervised learning when labeled data is limited.

### The Challenge Setup
Suppose we only have labels for **10% of the CIFAR-10 training set** (5,000 images).

* **Scenario A (Supervised Baseline):** Train a ResNet-18 directly from scratch on the 10% labeled data.
* **Scenario B (SimpleJEPA Pre-training + Linear Probe):** 
  1. Pre-train `SimpleJEPA` on the **100% unlabeled** CIFAR-10 dataset using context and target embeddings.
  2. Freeze `SimpleJEPA.context_encoder` and train a Linear Classifier on the 10% labeled dataset.

Let's compare test set accuracy!

In [ ]:
# --- Data Transformations for JEPA Views ---
class JEPATransform:
    """ Generates context and target spatial views for JEPA training """
    def __init__(self):
        self.ctx_transform = transforms.Compose([
            transforms.RandomResizedCrop(32, scale=(0.4, 0.8)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])
        self.tgt_transform = transforms.Compose([
            transforms.RandomResizedCrop(32, scale=(0.4, 0.8)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])
    def __call__(self, x):
        return self.ctx_transform(x), self.tgt_transform(x)

standard_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Load CIFAR-10 Datasets
full_unlabeled_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=JEPATransform())
full_labeled_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=standard_transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

# Create a 10% labeled subset (5,000 samples)
np.random.seed(42)
indices = np.random.choice(len(full_labeled_dataset), size=5000, replace=False)
labeled_subset = Subset(full_labeled_dataset, indices)

# DataLoaders
unlabeled_loader = DataLoader(full_unlabeled_dataset, batch_size=128, shuffle=True, num_workers=2)
labeled_loader = DataLoader(labeled_subset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

print(f"Full Unlabeled Dataset size: {len(full_unlabeled_dataset)}")
print(f"Limited Labeled Dataset size (10%): {len(labeled_subset)}")
print(f"Test Dataset size: {len(testset)}")

### Scenario A: Purely Supervised Baseline (10% Labeled Data)

In [ ]:
def evaluate_accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return 100.0 * correct / total

# ResNet-18 baseline built for CIFAR-10
supervised_model = models.resnet18(weights=None)
supervised_model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
supervised_model.maxpool = nn.Identity()
supervised_model.fc = nn.Linear(512, 10)
supervised_model = supervised_model.to(device)

optimizer = torch.optim.Adam(supervised_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print("Training Supervised Baseline on 10% labeled data for 10 epochs...")
supervised_model.train()
for epoch in range(10):
    for inputs, targets in labeled_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = supervised_model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

baseline_acc = evaluate_accuracy(supervised_model, test_loader)
print(f"➡️ Supervised Baseline Test Accuracy (10% labels): {baseline_acc:.2f}%")

### Scenario B: SimpleJEPA Pre-training (100% Unlabeled) + Linear Probing (10% Labeled)

In [ ]:
# 1. Initialize SimpleJEPA
jepa_model = SimpleJEPA(embed_dim=512).to(device)
optimizer_jepa = torch.optim.Adam(jepa_model.parameters(), lr=1e-3)

print("Pre-training SimpleJEPA on 100% UNLABELED CIFAR-10 (5 epochs demo)...")
jepa_model.train()
for epoch in range(5):
    running_loss = 0.0
    for (x_ctx, x_tgt), _ in unlabeled_loader:
        x_ctx, x_tgt = x_ctx.to(device), x_tgt.to(device)
        
        # Forward passes
        pred_s_y = jepa_model.forward_context(x_ctx)  # Online Context Encoder + Predictor
        s_y = jepa_model.forward_target(x_tgt)        # Target Encoder (EMA, frozen grad)
        
        # Calculate JEPA loss in representation space
        loss = jepa_loss(pred_s_y, s_y)
        
        optimizer_jepa.zero_grad()
        loss.backward()
        optimizer_jepa.step()
        
        # Update Target Encoder via EMA
        jepa_model.update_target_encoder(momentum=0.99)
        running_loss += loss.item()
        
    print(f"  Epoch {epoch+1}/5 - SimpleJEPA Loss: {running_loss/len(unlabeled_loader):.4f}")

# 2. Linear Probing: Freeze SimpleJEPA Context Encoder & Train Classifier on 10% Labeled Data
encoder = jepa_model.context_encoder
for param in encoder.parameters():
    param.requires_grad = False  # FREEZE BACKBONE!

linear_classifier = nn.Linear(512, 10).to(device)
optimizer_linear = torch.optim.Adam(linear_classifier.parameters(), lr=1e-2)

print("Training Linear Classifier on 10% labeled data (Frozen SimpleJEPA Backbone)...")
encoder.eval()
linear_classifier.train()
for epoch in range(10):
    for inputs, targets in labeled_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        with torch.no_grad():
            features = encoder(inputs)
        
        optimizer_linear.zero_grad()
        outputs = linear_classifier(features)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer_linear.step()

# Evaluate SimpleJEPA + Linear Probe
class JEPAProbeModel(nn.Module):
    def __init__(self, encoder, classifier):
        super().__init__()
        self.encoder = encoder
        self.classifier = classifier
    def forward(self, x):
        return self.classifier(self.encoder(x))

jepa_probe_model = JEPAProbeModel(encoder, linear_classifier).to(device)
jepa_acc = evaluate_accuracy(jepa_probe_model, test_loader)
print(f"➡️ SimpleJEPA Pre-trained Test Accuracy (10% labels): {jepa_acc:.2f}%")

### 📊 Final Comparison: Supervised vs. SimpleJEPA

In [ ]:
print('=' * 55)
print(f"10% Labeled Supervised Baseline Accuracy    : {baseline_acc:.2f}%")
print(f"10% Labeled SimpleJEPA Pre-training Accuracy: {jepa_acc:.2f}%")
print(f"Accuracy Gain with SimpleJEPA SSL          : +{jepa_acc - baseline_acc:.2f}%")
print('=' * 55)

# Plot Comparison Bar Chart
methods = ['Supervised Baseline\n(10% Labels)', 'SimpleJEPA SSL + Linear Probe\n(10% Labels)']
accuracies = [baseline_acc, jepa_acc]

plt.figure(figsize=(8, 5))
plt.bar(methods, accuracies, color=['#e74c3c', '#2ecc71'], width=0.4)
plt.ylabel('Test Accuracy (%)')
plt.title('CIFAR-10 Accuracy: Supervised Baseline vs. SimpleJEPA (10% Labeled Data)')
plt.ylim(0, 100)
for i, v in enumerate(accuracies):
    plt.text(i, v + 2, f"{v:.2f}%", ha='center', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()